In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [ ]:
my_layer=Dense(units=1,input_shape=[1])
my_model=Sequential([my_layer])

opt='SGD'
loss_function='mean_squared_error'
my_model.compile(optimizer=opt,loss=loss_function)

xs=np.array([-1.0,0.0,1.0,2.0,3.0,4.0],dtype=float)
ys=np.array([1.0,3.0,5.0,7.0,9.0,11.0],dtype=float)

my_model.fit(xs,ys,epochs=300)



In [ ]:
test_x=np.array([10.0])
print(my_model.predict(test_x))

print("the parameters of the model are  :", my_model.get_weights())


In [ ]:
#Saving the model 
export_dir='saved_model/CV_practice'
tf.saved_model.save(my_model,export_dir)


In [ ]:
#converting into tflite model:

converter=tf.lite.TFLiteConverter.from_saved_model(export_dir)
tflite_model=converter.convert()  #converting tf to tflite model 



In [ ]:
#Saving the tflite model to disk
import pathlib

#creating a file path for our tflite model in which it will be saved 
tflite_model_file=pathlib.Path('model.tflite')  #always save the tflite models with the extension .tflite


#writing the information of our tflite model into the file tflite_model_file
print("the size of the tflite model file in bytes is :",tflite_model_file.write_bytes(tflite_model))
#writing the information of our tflite model into the file tflite_model_file
 #it also tell you the size of the model 


In [ ]:
#to check wether the model has lost its accuracy or not we have to test it on hardware but first we test it on software using:

#Model inference using tensorflow interpreter :

#1. Allocate the tensors () to the loaded model
interpreter=tf.lite.Interpreter(model_content=tflite_model) #think of it as an array 
interpreter.allocate_tensors()

#2. Now we need the information about the input and output interpreter so that we can test the accuracy of the model:

input_details=interpreter.get_input_details()
output_details=interpreter.get_output_details()
print(input_details)
print(output_details)

#the above is used to get the index of the input and the output so that it can later be used to feed the test input and get the test output
#it can also be used to know about the shapes of the input and output 


In [ ]:
#using the  test data to run inference

to_predict=np.array([10.0],dtype=np.float32).reshape([1,1]) #use the same dtype as the train input

#setting the value to predict at the correct interpreter index of the input tensor
interpreter.set_tensor(input_details[0]['index'],to_predict)

#running inference on to predict value (getting the output at the set index) similar to model.predict
interpreter.invoke()

#reading the output which is stored at the appropriate index location
tflite_results=interpreter.get_tensor(output_details[0]['index'])
print(f'the predicted value is : {tflite_results}')

#added some changes to the fil